# 01 — Reference genotypes and blood embeddings

Welcome to the ATTICUS tutorial, good to see you! Here, we'll guide you through an example using OneK1K. OneK1K data is freely available, while ROSMAP is not (sorry).

After following the [README installation steps](../README.md#getting-started), we'll prepare the genotype reference and any missing blood embeddings.

Before starting, download the prepared expression file from the [OneK1K dataset on Hugging Face](https://huggingface.co/datasets/Padraig7/atticus-onek1k) and place it at **`data/onek1k/expression.h5ad`**. The [expression format](../docs/command_reference.md#expression-file) has the details.

The same [Hugging Face dataset](https://huggingface.co/datasets/Padraig7/atticus-onek1k) provides **ready-made OneK1K embeddings**. Place them under `data/onek1k/embeddings/`, keeping `reference/REFERENCE.npz`, `individuals/<sample>.npz`, and `individuals/samples.txt`. Once expression and `REFERENCE.npz` are in place, you may **skip VariantFormer installation and inference for reference training** and head straight to [notebook 02](02_onek1k_reference_training.ipynb).

Reference training needs no OneK1K genotypes. Distillation uses personalized predictions on the independent 503-person genotype reference under `data/geuvadis/`. Those individuals need their own embeddings; the OneK1K individual archives belong to a different cohort.


In [2]:
from pathlib import Path
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'pyproject.toml').is_file() and (path / 'src/training/run.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the ATTICUS repository.')
for directory in (ROOT, ROOT / 'notebooks'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial import ROOT, PATHS, run, uv

EXECUTE = True  # Set to False to preview commands without running them.
print('Repository:', ROOT)
print('Mode:', 'execute commands' if EXECUTE else 'preview commands')


Repository: /local1/scratch/pstyll/test/atticus
Mode: execute commands


## 1. Prepare the genotype reference

The script downloads GRCh38 autosomes from [Zenodo record 16538482](https://zenodo.org/records/16538482) and selects the 503 individuals listed in `data/data-for-the-paper/1000G_EUR_503.txt`. It creates biallelic PLINK files and one indexed VCF per individual, checks download checksums, and resumes completed chromosomes. No expression measurements from this panel are needed.

This is a good moment to check disk space: the source download is about 27 GB, before adding PLINK files, individual VCFs, and embeddings. You'll need the command-line tools listed in the [README](../README.md#getting-started).


In [3]:
run(['bash', 'setup.sh', '--geuvadis'], execute=EXECUTE)

expected = [PATHS['genotypes'] / f'chr{chrom}.GRCh38.biallelic.{suffix}'
            for chrom in range(1, 23) for suffix in ('bed', 'bim', 'fam')]
print('Prepared PLINK files:', sum(path.is_file() for path in expected), '/', len(expected))
if PATHS['samples'].is_file():
    from src.preprocessing.geuvadis import read_ids
    samples = read_ids(PATHS['samples'])
    individuals = ROOT / 'data/geuvadis/individuals'
    print('Reference individuals:', len(samples))
    print('Complete individual VCF/index pairs:', sum(
        (individuals / f'{sample}.vcf.gz').is_file() and (individuals / f'{sample}.vcf.gz.tbi').is_file()
        for sample in samples))
else:
    print('The sample list is created by setup.')


Running: bash setup.sh --geuvadis


Missing plink2. Install it or load its module so it is on PATH.


CalledProcessError: Command '['bash', 'setup.sh', '--geuvadis']' returned non-zero exit status 1.

## 2. Generate missing embeddings with VariantFormer (optional)

Skip inference for embeddings you've downloaded. If both the reference archive and the Geuvadis individual archives are ready, skip this whole section, including the VariantFormer installation.

Here's where the GPU gets involved. Use Linux x86_64 with a supported CUDA 12 environment. The installer downloads the protein-coding checkpoint, tokenizer, GRCh38 reference, annotation, and regulatory regions. Reference and personalized embeddings must use the **same blood-conditioned protein-coding model**.

Both commands generate embeddings for VariantFormer's **autosomal protein-coding genes**. Training aligns the reference embeddings with the genes in your expression file. Prediction, distillation, and TWAS each use the genes available in their inputs; there is no gene-list file to prepare.

Skip the installation cell if the dependencies and assets are already in place. Set `EXECUTE = True` on the machine with the required tools and CUDA environment.


In [ ]:
run(['bash', 'setup.sh', '--variantformer'], execute=EXECUTE)


### Reference genome embeddings

If `data/onek1k/embeddings/reference/REFERENCE.npz` is available, we'll reuse it. Otherwise, the command writes `data/embeddings/reference/REFERENCE.npz`. To put it elsewhere, set `PATHS["reference"]` to a file named `REFERENCE.npz`; generation uses its parent directory.

New NPZs include tissue and model metadata for compatibility checks. Supplied embeddings keep unrecorded details as unknown; the loader checks known fields and warns about unknown ones.

These frozen embeddings describe the unmodified reference genome and are shared across OneK1K donors. Personalized embeddings come next.


In [ ]:
reference = PATHS['reference']
if reference.is_file():
    print('Reusing reference embeddings:', reference)
else:
    if reference.name != 'REFERENCE.npz':
        raise ValueError('Set PATHS["reference"] to a file named REFERENCE.npz.')
    run(uv('variantformer', 'src/preprocessing/embeddings.py', '--tissue', 'blood', '--reference',
           '--output', reference.parent), execute=EXECUTE)


### Personalized embeddings for every reference individual

If the Geuvadis embeddings are already present in `data/onek1k/embeddings/geuvadis/`, skip generation. Both `<sample>.npz` archives and `<sample>/` directories containing `features.npy`, `ensids.npy`, `chroms.npy`, and `tss.npy` are supported. Otherwise, use the individual VCFs prepared above. This is the GPU-intensive part, so a batch job may be more comfortable than keeping a notebook open.

VariantFormer fetches gene-specific assets as needed. The first job needs network access or a populated asset cache; installing the base assets alone doesn't make it fully offline.

The script saves one `<sample>.npz` at a time and resumes matching completed archives. Each holds features, Ensembl IDs, chromosomes, TSS, sample ID, and tissue.


In [ ]:
from src.preprocessing.embeddings import personalized_files, personalized_path
from src.preprocessing.geuvadis import read_ids

samples = read_ids(PATHS['samples']) if PATHS['samples'].is_file() else []
def embeddings_ready():
    return sum(all(path.is_file() for path in personalized_files(
        personalized_path(PATHS['personalized'], sample))) for sample in samples)

if not samples or embeddings_ready() < len(samples):
    run(uv('variantformer', 'src/preprocessing/embeddings.py', '--tissue', 'blood',
           '--samples', PATHS['samples'], '--vcf-dir', ROOT / 'data/geuvadis/individuals',
           '--output', PATHS['personalized']), execute=EXECUTE)

if samples:
    print('Personalized embeddings present:', embeddings_ready(), '/', len(samples))


## Ready for training

You should now have OneK1K expression, blood reference embeddings, reference-panel genotypes, and personalized blood embeddings. You can start notebook 02 as soon as expression and reference embeddings are ready; personalized embeddings can run separately before notebook 03.

`PATHS["personalized"]` points to `data/onek1k/embeddings/geuvadis/` for the independent 503-person Geuvadis panel, matching the distillation sample list and genotypes. Downloaded OneK1K personalized archives live separately under `data/onek1k/embeddings/individuals/`.

The blood embeddings are shared across cell types. In notebook 02, choose one cell type for the demonstration; we default to **memory B cells**.

Next up: [02 — OneK1K reference ensemble training](02_onek1k_reference_training.ipynb).
